In [25]:
import pandas as pd

data = pd.read_csv("CP_all_EveBio.csv")
data

,InChIKey,ADRA1A_Antagonist,ADRA2A_Antagonist,ADRA2B_Antagonist,CAR_Antagonist,ERa_Antagonist,ERb_Antagonist,GPR119_Antagonist,GR_Agonist,HCTR1_Antagonist,...,Nuclei_Texture_Variance_DNA_5_0,Nuclei_Texture_Variance_ER_10_0,Nuclei_Texture_Variance_ER_3_0,Nuclei_Texture_Variance_ER_5_0,Nuclei_Texture_Variance_Mito_10_0,Nuclei_Texture_Variance_Mito_3_0,Nuclei_Texture_Variance_Mito_5_0,Nuclei_Texture_Variance_RNA_10_0,Nuclei_Texture_Variance_RNA_3_0,Nuclei_Texture_Variance_RNA_5_0
0,VABCILAOYCMVPS-UHFFFAOYSA-N,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,-0.066118,0.057464,0.010237,0.029738,0.041859,0.036450,0.044164,-0.130869,-0.074984,-0.082211
1,RZVAJINKPMORJF-UHFFFAOYSA-N,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.063601,-0.031328,-0.041462,-0.033558,-0.039537,-0.047899,-0.057852,-0.037782,-0.027523,-0.019829
2,BZKPWHYZMXOIDC-UHFFFAOYSA-N,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,-0.217802,-0.117676,-0.135966,-0.130286,-0.068340,-0.050347,-0.082794,0.080201,0.045306,0.025637
3,VGZSUPCWNCWDAN-UHFFFAOYSA-N,1.0,1.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,...,0.034478,-0.008062,0.003948,-0.007792,-0.016776,-0.001372,0.010641,0.019953,-0.004199,0.001124
4,NDAUXUAQIAJITI-UHFFFAOYSA-N,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,-0.161761,-0.090080,-0.123328,-0.093332,-0.027687,-0.105908,-0.098367,-0.066008,-0.129280,-0.118737
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
264,PJVWKTKQMONHTI-UHFFFAOYSA-N,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,-0.089445,-0.080948,-0.079638,-0.057819,-0.004393,-0.069130,-0.053182,-0.132504,-0.120575,-0.125685
265,BLGXFZZNTVWLAY-SCYLSFHTSA-N,1.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,...,-0.280346,-0.046786,-0.121671,-0.103115,-0.024106,-0.098937,-0.059982,-0.027537,-0.068917,-0.039486
266,HBOMLICNUCNMMY-XLPZGREQSA-N,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,...,-0.138426,-0.013285,-0.060643,-0.051464,0.011607,-0.019032,-0.003195,0.029683,0.037218,0.033849
267,UBQNRHZMVUUOMG-UHFFFAOYSA-N,0.0,0.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,...,-0.135122,-0.022240,-0.039310,-0.015211,0.025192,0.027944,0.037482,-0.082967,-0.069643,-0.035444


In [39]:
CellProfiler = data.columns.to_list()[28:]

In [40]:
assays_list = data.columns.to_list()[1:25]
len(assays_list)
assays_list

['ADRA1A_Antagonist',
 'ADRA2A_Antagonist',
 'ADRA2B_Antagonist',
 'CAR_Antagonist',
 'ERa_Antagonist',
 'ERb_Antagonist',
 'GPR119_Antagonist',
 'GR_Agonist',
 'HCTR1_Antagonist',
 'PPARa_Antagonist',
 'PPARg_Antagonist',
 'PR_Agonist',
 'PR_Antagonist',
 'PXR_Agonist',
 'RARa_Antagonist',
 'RARb_Antagonist',
 'RARg_Antagonist',
 'RORa_Antagonist',
 'RORb_Antagonist',
 'RORg_Antagonist',
 'RXRa_Antagonist',
 'RXRb_Antagonist',
 'RXRg_Antagonist',
 'SF1_Antagonist']

In [41]:
len(assays_list)

24

In [42]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score
import pandas as pd
from tqdm import tqdm
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.model_selection import RandomizedSearchCV
import os

In [43]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    matthews_corrcoef,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
    auc,
    cohen_kappa_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict, RandomizedSearchCV
from sklearn.utils import class_weight
import pandas as pd
from tqdm import tqdm
from sklearn.utils import shuffle

# Function to calculate metrics
def calculate_metrics(y_true, y_pred_proba, threshold=0.5):
    y_pred_opt = (y_pred_proba >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred_opt).ravel()
    sensitivity = tp / (tp + fn)
    specificity = tn / (tn + fp)
    balanced_accuracy = (sensitivity + specificity) / 2
    mcc = matthews_corrcoef(y_true, y_pred_opt)
    kappa = cohen_kappa_score(y_true, y_pred_opt)
    
    # AUCPR
    precision, recall, _ = precision_recall_curve(y_true, y_pred_proba)
    aucpr = auc(recall, precision)
    
    # Cohen's Kappa at threshold 0.50
    kappa_at_50 = cohen_kappa_score(y_true, (y_pred_proba >= 0.50).astype(int))
    
    return {
        "Threshold": threshold,
        "Balanced Accuracy": balanced_accuracy,
        "Sensitivity": sensitivity,
        "Specificity": specificity,
        "MCC": mcc,
        "AUCPR": aucpr,
        "Cohen's Kappa": kappa,
        "Cohen's Kappa @ 0.50": kappa_at_50,
        "Confusion Matrix": {"TN": tn, "FP": fp, "FN": fn, "TP": tp},
        "TN": tn,
        "FP": fp,
        "FN": fn,
        "TP": tp,
    }

# Function to find the best threshold based on Cohen's Kappa
def find_best_threshold_for_kappa(y_true, y_pred_proba):
    thresholds = np.linspace(0, 1, 100)
    best_kappa = -1
    best_threshold = 0.5
    for threshold in thresholds:
        y_pred = (y_pred_proba >= threshold).astype(int)
        kappa = cohen_kappa_score(y_true, y_pred)
        if kappa > best_kappa:
            best_kappa = kappa
            best_threshold = threshold
    return best_threshold, best_kappa

# Initialize results
all_auc_scores = []

# Stratified splitting setup
stratified_kfold = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# Define the two sets of features
feature_sets = {
    "CellCount": ["Cells_Number_Object_Number"],  # Feature set cell count
    "CellProfiler": CellProfiler  # Feature set CP
}

# Loop through each assay to create individual models
for assay in tqdm(assays_list):
    
    # Filter out rows with missing values for the current assay
    assay_data = data[~data[assay].isna()].reset_index(drop=True)
    print(assay_data[assay].value_counts())
    
    # Extract labels
    y = assay_data[assay].astype(int)

    # Check for class imbalance
    if len(np.unique(y)) < 2:
        print(f"Skipping assay {assay} due to lack of positive/negative classes")
        continue
    
    # Calculate class weights
    class_weights = class_weight.compute_class_weight('balanced', classes=np.unique(y), y=y)
    class_weight_dict = {0: class_weights[0], 1: class_weights[1]}
    
    rf_param_grid = {
        'n_estimators': [100, 200, 300],
        'max_depth': [5, 10, None],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'class_weight': ['balanced']
    }
    
    logreg_param_grid = {
        'C': [0.01, 0.1, 1, 10, 100],
        'penalty': ['l1', 'l2'],
        'solver': ['liblinear']
    }
    
    for feature_set_name, feature_columns in feature_sets.items():
        
        # Extract features for the current feature set
        X = assay_data[feature_columns]

        for fold, (train_idx, test_idx) in enumerate(stratified_kfold.split(X, y)):
            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
                 
            # Determine the model type based on the number of features
            if len(feature_columns) == 1:
                # Use Logistic Regression
                model = LogisticRegression(class_weight='balanced', random_state=42)
                param_grid = logreg_param_grid
            else:
                # Use Random Forest
                model = RandomForestClassifier(random_state=42, n_jobs=-1)
                param_grid = rf_param_grid

            # Perform RandomizedSearchCV
            random_search = RandomizedSearchCV(
                estimator=model,
                param_distributions=param_grid,
                n_iter=10,
                scoring='roc_auc',
                cv=3,
                n_jobs=-1,
                verbose=2,
                random_state=42
            )
            random_search.fit(X_train, y_train)

            # Best model
            best_model = random_search.best_estimator_

            # Generate OOB predictions for training data
            oob_pred_proba = cross_val_predict(
                best_model,
                X_train,
                y_train,
                cv=stratified_kfold,
                method='predict_proba'
            )[:, 1]
            
            # Optimize threshold using Cohen's Kappa
            opt_threshold, best_kappa = find_best_threshold_for_kappa(y_train, oob_pred_proba)
            
            # Test set predictions
            y_pred_proba = best_model.predict_proba(X_test)[:, 1]
            test_metrics = calculate_metrics(y_test, y_pred_proba, threshold=opt_threshold)
            
            # Random Y-scrambling
            y_train_scrambled = shuffle(y_train, random_state=fold)
            best_model.fit(X_train, y_train_scrambled)
            y_pred_proba_scrambled = best_model.predict_proba(X_test)[:, 1]
            scrambled_metrics = calculate_metrics(y_test, y_pred_proba_scrambled, threshold=0.50)
            
            # Collect results
            all_auc_scores.append({
                "Feature Set": feature_set_name,
                "Fold": fold + 1,
                "Task": assay,
                "AUC": roc_auc_score(y_test, y_pred_proba),
                "Optimal Threshold": opt_threshold,
                "Optimal Kappa": best_kappa,
                "Kappa": test_metrics["Cohen's Kappa"],
                "Kappa @ 0.50": test_metrics["Cohen's Kappa @ 0.50"],
                "Balanced Accuracy": test_metrics["Balanced Accuracy"],
                "AUCPR": test_metrics["AUCPR"],
                "Sensitivity": test_metrics["Sensitivity"],
                "Specificity": test_metrics["Specificity"],
                "MCC": test_metrics["MCC"],
                "TP": test_metrics["TP"],
                "TN": test_metrics["TN"],
                "FP": test_metrics["FP"],
                "FN": test_metrics["FN"],
                "Total Actives Train": sum(y_train),
                "Total Inactives Train": len(y_train) - sum(y_train),
                "Total Compounds Train": len(y_train),
                "Total Actives Test": sum(y_test),
                "Total Inactives Test": len(y_test) - sum(y_test),
                "Total Compounds Test": len(y_test),
                "Scrambled Kappa": scrambled_metrics["Cohen's Kappa"],
                "Scrambled Balanced Accuracy": scrambled_metrics["Balanced Accuracy"],
            })
            print(f"Assay: {assay}, Fold: {fold + 1}, AUC: {roc_auc_score(y_test, y_pred_proba):.3f}")
            # Convert results to DataFrame and save
            auc_df = pd.DataFrame(all_auc_scores)
            auc_df.to_csv("auc_scores_with_rf_kappa_thresholding_detailed.csv", index=False)


# Convert results to DataFrame and save
auc_df = pd.DataFrame(all_auc_scores)
auc_df.to_csv("auc_scores_with_rf_kappa_thresholding_detailed.csv", index=False)


  0%|          | 0/24 [00:00<?, ?it/s]

0.0    169
1.0     42
Name: ADRA1A_Antagonist, dtype: int64
Fitting 3 folds for each of 10 candidates, totalling 30 fits
Assay: ADRA1A_Antagonist, Fold: 1, AUC: 0.482
Fitting 3 folds for each of 10 candidates, totalling 30 fits
Assay: ADRA1A_Antagonist, Fold: 2, AUC: 0.636
Fitting 3 folds for each of 10 candidates, totalling 30 fits
Assay: ADRA1A_Antagonist, Fold: 3, AUC: 0.571
Fitting 3 folds for each of 10 candidates, totalling 30 fits
Assay: ADRA1A_Antagonist, Fold: 1, AUC: 0.476
Fitting 3 folds for each of 10 candidates, totalling 30 fits
Assay: ADRA1A_Antagonist, Fold: 2, AUC: 0.615
Fitting 3 folds for each of 10 candidates, totalling 30 fits


  4%|▍         | 1/24 [00:17<06:36, 17.24s/it]

Assay: ADRA1A_Antagonist, Fold: 3, AUC: 0.649
0.0    209
1.0     25
Name: ADRA2A_Antagonist, dtype: int64
Fitting 3 folds for each of 10 candidates, totalling 30 fits
Assay: ADRA2A_Antagonist, Fold: 1, AUC: 0.500
Fitting 3 folds for each of 10 candidates, totalling 30 fits
Assay: ADRA2A_Antagonist, Fold: 2, AUC: 0.500
Fitting 3 folds for each of 10 candidates, totalling 30 fits
Assay: ADRA2A_Antagonist, Fold: 3, AUC: 0.500
Fitting 3 folds for each of 10 candidates, totalling 30 fits
Assay: ADRA2A_Antagonist, Fold: 1, AUC: 0.673
Fitting 3 folds for each of 10 candidates, totalling 30 fits
Assay: ADRA2A_Antagonist, Fold: 2, AUC: 0.663
Fitting 3 folds for each of 10 candidates, totalling 30 fits


  8%|▊         | 2/24 [00:28<04:58, 13.57s/it]

...

Assay: RXRb_Antagonist, Fold: 3, AUC: 0.601
0.0    213
1.0     37
Name: RXRg_Antagonist, dtype: int64
Fitting 3 folds for each of 10 candidates, totalling 30 fits
Assay: RXRg_Antagonist, Fold: 1, AUC: 0.348
Fitting 3 folds for each of 10 candidates, totalling 30 fits
Assay: RXRg_Antagonist, Fold: 2, AUC: 0.479
Fitting 3 folds for each of 10 candidates, totalling 30 fits
Assay: RXRg_Antagonist, Fold: 3, AUC: 0.500
Fitting 3 folds for each of 10 candidates, totalling 30 fits
Assay: RXRg_Antagonist, Fold: 1, AUC: 0.692
Fitting 3 folds for each of 10 candidates, totalling 30 fits
Assay: RXRg_Antagonist, Fold: 2, AUC: 0.719
Fitting 3 folds for each of 10 candidates, totalling 30 fits


 96%|█████████▌| 23/24 [04:40<00:11, 11.65s/it]

Assay: RXRg_Antagonist, Fold: 3, AUC: 0.744
0.0    217
1.0     37
Name: SF1_Antagonist, dtype: int64
Fitting 3 folds for each of 10 candidates, totalling 30 fits
Assay: SF1_Antagonist, Fold: 1, AUC: 0.500
Fitting 3 folds for each of 10 candidates, totalling 30 fits
Assay: SF1_Antagonist, Fold: 2, AUC: 0.504
Fitting 3 folds for each of 10 candidates, totalling 30 fits
Assay: SF1_Antagonist, Fold: 3, AUC: 0.500
Fitting 3 folds for each of 10 candidates, totalling 30 fits
Assay: SF1_Antagonist, Fold: 1, AUC: 0.735
Fitting 3 folds for each of 10 candidates, totalling 30 fits
Assay: SF1_Antagonist, Fold: 2, AUC: 0.707
Fitting 3 folds for each of 10 candidates, totalling 30 fits


100%|██████████| 24/24 [04:53<00:00, 12.23s/it]

Assay: SF1_Antagonist, Fold: 3, AUC: 0.595
